# Phase O — Tuning, and the null it returned

Phase O tunes the run, not the model. Three knobs — learning rate, warmup, epoch budget —
against one metric, `reference` (held-out 2024 unweighted log loss per scored row). The
architecture, the arm, the loss, the features and the tensor build are frozen exactly as
D.10 shipped them. `docs/phase-o-spec.md` is the authority for everything below and was
written before any selection run existed.

**The result is a null.** The incumbent — learning rate 1e-3, no warmup, the setting held
fixed across all 119 pre-O runs — was already the best of the seven arms. No arm was
promoted. Phase M and the 2025 refit run on the D.10 build unchanged.

**Claim 1 is never read here.** Not as a tiebreak, not as a sanity check.
`src/analysis/hyperparameter_tuning_select.py` does not import `claim1_eval`, and
`tests/test_hyperparameter_tuning_select.py` asserts that it does not. The `claim1_read`
flag printed below is the artifact's own record of that.

**2025 is sealed.** No cell here reads, scores, or refits on it.

This notebook **reads committed artifacts and recomputes nothing**. Every number is
produced by `src/analysis/hyperparameter_tuning_select.py` and the sweep ledger; this
notebook only arranges them.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

## Configuration

Artifact locations. `selection.json` is the verdict; `sweep_log.csv` is the run ledger the
verdict was computed from, and carries the knob settings each arm actually ran at.

In [2]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

PHASE_O_DIR = REPO_ROOT / "results/hyperparameter_tuning"
LEDGER = REPO_ROOT / "results/model_v1/sweep_log.csv"

STAGE = "selection"
INCUMBENT = "lr1e3"

selection = json.loads((PHASE_O_DIR / "selection.json").read_text())
arms = pd.read_csv(PHASE_O_DIR / "selection.csv")
ledger = pd.read_csv(LEDGER).query("stage == @STAGE")

print(f"{len(ledger)} ledger rows at stage '{STAGE}', statuses: {sorted(ledger.status.unique())}")

14 ledger rows at stage 'selection', statuses: ['ok']


## The rule, as it was fixed before the runs

Printed first, before a single arm score is read. `margin_ses` is the promotion bar;
`noise_floor_sd` is the across-seed standard deviation of `reference` for the D.10
`rebuild` baseline — five seeds, same architecture, same build — read from the ledger at
run time rather than hardcoded.

In [3]:
print(json.dumps(selection["rule"], indent=2))

{
  "metric": "reference (held-out unweighted log loss per scored row)",
  "incumbent": "lr1e3",
  "margin_ses": 2.0,
  "noise_floor_sd": 0.00010416333327999322,
  "noise_floor_source": "rebuild/baseline",
  "noise_floor_n_seeds": 5,
  "claim1_read": false
}


## The build guard

The incumbent arm re-runs the D.10 baseline recipe at a different stage, so a run on the
wrong data build would be in different units while every column still lined up. The guard
here is stronger than the 4-sd tolerance the spec pre-registered: the incumbent reproduces
the rebuild baseline **exactly** on the seeds the two stages share, so the drift is not
merely inside tolerance, it is a bit-level match on shared seeds.

In [4]:
guard = selection["guard"]
print(json.dumps(guard, indent=2))
assert guard["passed"], "selection is refused if the build guard fails"

{
  "incumbent_mean": 1.025845,
  "rebuild_baseline_mean": 1.0258,
  "drift": 4.4999999999850715e-05,
  "tolerance": 0.00041665333311997286,
  "drift_is_informative": false,
  "basis": "exact reproduction on shared seeds",
  "shared_seeds": [
    0,
    1
  ],
  "independent_of_noise_floor": false,
  "reproduces_shared_seeds": true,
  "mismatched_seeds": [],
  "passed": true
}


## The grid is complete, so the mean is over what it claims to be

`sweep.queue` is config-major, so a night that runs short drops whole arms off the end
rather than thinning every arm evenly — and a mean over the three arms that finished is
indistinguishable from a mean over seven. A partial grid returns `incomplete_grid` and is
refused, not averaged.

In [5]:
print(json.dumps(selection["grid"], indent=2))

{
  "expected_arms": [
    "lr1e3",
    "lr1e3_warm",
    "lr1e3_warm2k",
    "lr3e3",
    "lr3e3_warm",
    "lr3e4",
    "lr3e4_warm"
  ],
  "missing_arms": [],
  "underpowered_arms": [],
  "unexpected_arms": [],
  "min_seeds_per_arm": 2,
  "complete": true
}


## The factorial

Seven arms: the 3×2 learning-rate × warmup factorial, plus `lr1e3_warm2k` — a longer
2000-step warmup added on the runner-up cell, since `lr1e3_warm` was the only arm on the
correct side of the incumbent and a one-epoch warmup is the shortest schedule that exists.
Two seeds each. Positive `margin_in_ses` means better than the incumbent.

In [6]:
knobs = (ledger.groupby("config")[["lr", "warmup_steps"]].first()
         .join(ledger.groupby("config")["best_epoch"].apply(list).rename("best_epochs")))

table = (pd.DataFrame(selection["arms"]).T
         .rename_axis("config").reset_index()
         .merge(knobs.reset_index(), on="config")
         .sort_values("margin_in_ses", ascending=False)
         .loc[:, ["config", "lr", "warmup_steps", "n_seeds", "reference_mean",
                  "reference_sd", "margin_in_ses", "promotable", "best_epochs"]])
table

,config,lr,warmup_steps,n_seeds,reference_mean,reference_sd,margin_in_ses,promotable,best_epochs
1,lr1e3_warm,0.0010,719,2,1.02576,0.000127,0.816026,False,"[17, 18]"
2,lr1e3_warm2k,0.0010,2000,2,1.0258,0.000226,0.432014,False,"[17, 18]"
0,lr1e3,0.0010,0,2,1.025845,0.000007,0.0,False,"[17, 17]"
4,lr3e3_warm,0.0030,719,2,1.026,0.00041,-1.488048,False,"[9, 9]"
3,lr3e3,0.0030,0,2,1.02613,0.000141,-2.736088,False,"[8, 9]"
5,lr3e4,0.0003,0,2,1.02692,0.000721,-10.32033,False,"[19, 27]"
6,lr3e4_warm,0.0003,719,2,1.02694,0.000523,-10.512336,False,"[19, 27]"


Two things to read off this table beyond the verdict.

`best_epoch` never approaches the 50-epoch cap in any arm — the highest is 27, at the
slowest learning rate. Early stopping already governs the epoch budget, which is why the
spec observes that knob rather than sweeping it.

The two arms at lr 3e-4 sit **10 standard errors worse** than the incumbent. That is the
signal the metric is licensed to detect: an undertrained run is worse on its own training
objective. The metric is used here to detect undertraining, never to rank converged models
against each other — Phase E measured a claim-1/likelihood rank correlation of 0.000 across
the seven D.10 arms, and ranking on likelihood is barred for that reason.

## The pre-registered expectation, and the outcome

The expectation carried into this pass: the warmup arm does not clear 2 SE. It did not.

In [7]:
best_challenger = table.query("config != @INCUMBENT").iloc[0]
print(f"verdict:            {selection['verdict']}")
print(f"winner:             {selection['winner']}")
print(f"challengers tested: {selection['n_challengers_tested']}")
print(f"bar:                {selection['rule']['margin_ses']} SE")
print(f"best challenger:    {best_challenger.config} at {best_challenger.margin_in_ses:.2f} SE")
assert selection["verdict"] == "incumbent_stands"
assert not any(a["promotable"] for a in selection["arms"].values())

verdict:            incumbent_stands
winner:             lr1e3
challengers tested: 6
bar:                2.0 SE
best challenger:    lr1e3_warm at 0.82 SE


## Why the confirmation step is moot rather than skipped

The spec's promotion rule carries a second stage: an arm that clears 2 SE on the two-seed
screen is `tuned_pending_confirmation` and must be re-run at five seeds before it becomes
`tuned`. A two-seed screen selects; it never concludes.

The screen ran at `--seeds 2`, not 5, and that is a decision rather than a shortcut.
`--seeds 5` would have backfilled eighteen runs onto six already-concluded configs
**including the incumbent**, shrinking `se = floor_sd × sqrt(1/n_arm + 1/n_inc)` after the
results were seen — moving a pre-registered gate post hoc.

The question that makes the depth moot is what the same observed margins would score at
greater depth. Holding each arm's observed mean fixed and varying only the seed counts:

In [8]:
import math

FLOOR = selection["rule"]["noise_floor_sd"]
BAR = selection["rule"]["margin_ses"]


def in_ses(margin, n_arm, n_inc):
    """Re-score an observed margin at hypothetical seed depths, per the frozen rule."""
    return margin / (FLOOR * math.sqrt(1 / n_arm + 1 / n_inc))


depths = [(2, 2), (5, 2), (5, 5)]
rows = []
for name in ["lr1e3_warm", "lr1e3_warm2k"]:
    margin = selection["arms"][name]["margin_vs_incumbent"]
    rows.append({"config": name, "margin": margin,
                 **{f"{a}v{i} SE": round(in_ses(margin, a, i), 2) for a, i in depths}})
depth = pd.DataFrame(rows)

weakest = min(r["margin"] for r in rows)
need = math.ceil(2 / (weakest / (FLOOR * BAR)) ** 2)
print(depth.to_string(index=False))
print(f"\nseeds per arm needed for the weaker arm to reach {BAR} SE, "
      f"at equal depth: {need}")

      config   margin  2v2 SE  5v2 SE  5v5 SE
  lr1e3_warm 0.000085    0.82    0.98    1.29
lr1e3_warm2k 0.000045    0.43    0.52    0.68

seeds per arm needed for the weaker arm to reach 2.0 SE, at equal depth: 43


Neither arm comes close at the confirmation depth the rule pre-registered. Item 3 of the
Pass B spec — adopt the winner, rerun A2, supersede the config — is therefore **dead, not
deferred**: there is no winner to adopt and no further GPU time that would produce one.

**What this does not say.** The standard error shrinks without bound as seeds are added, so
a fixed nonzero margin clears any fixed bar eventually — the seed count printed above is
what it would take here, and it is far outside the compute budget. The honest statement is
not "warmup cannot help"; it is that at the depth this project pre-registered, and against
the run-to-run noise this build actually carries, warmup is indistinguishable from no
warmup. Note also that both warmup arms' own across-seed `reference_sd` runs above the
`rebuild` noise floor the rule denominates in — with two seeds that is a two-point spread
and not an estimate, but it means the rule's SE is, if anything, generous to the
challengers.

## The final config

Unchanged since Phase D.10. This is the config the 2025 out-of-time refit runs on.

In [9]:
final = ledger.query("config == @INCUMBENT")
print(f"final config: {INCUMBENT}")
print(f"  lr:            {final.lr.iloc[0]}")
print(f"  warmup_steps:  {final.warmup_steps.iloc[0]}")
print(f"  data_dir:      {final.data_dir.iloc[0]}")
print(f"  best_epoch:    {sorted(final.best_epoch)} (cap 50)")
print(f"  reference:     {sorted(final.reference)}")
print("  batch_size / weight_decay: quarantined by phase-o-spec.md §4, 8192 / 1e-2")

final config: lr1e3
  lr:            0.001
  warmup_steps:  0
  data_dir:      data/processed/phase_d5
  best_epoch:    [17, 17] (cap 50)
  reference:     [1.02584, 1.02585]
  batch_size / weight_decay: quarantined by phase-o-spec.md §4, 8192 / 1e-2
